***

Profesor: Gonzalo A. Ruz, PhD

Curso: Deep Learning

***

# Primer MLP en Keras: clasificación multiclase con MNIST

## Objetivo

En este notebook construiremos y entrenaremos nuestro primer perceptrón multicapa (MLP) usando Keras.

Al finalizar, deberías poder:

- cargar y explorar un dataset,
- definir un modelo secuencial,
- compilarlo con una función de pérdida y métricas,
- entrenarlo con `fit`,
- evaluar su desempeño,
- interpretar curvas de entrenamiento y validación.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

## 1. Cargar y entender los datos

Usaremos el dataset [MNIST](https://en.wikipedia.org/wiki/MNIST_database) , que contiene imágenes en escala de grises de dígitos escritos a mano (0 al 9).

Cada imagen tiene tamaño 28×28 y pertenece a una de 10 clases.

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. De imagen a vector

Como vamos a usar una red densa (MLP), cada imagen de 28×28 se convertirá en un vector de 784 valores.

Además:

- normalizaremos los píxeles al rango [0, 1],
- codificaremos las etiquetas en formato one-hot.

In [ ]:
X_train = X_train.reshape((60000, 28 * 28)).astype("float32") / 255
X_test = X_test.reshape((10000, 28 * 28)).astype("float32") / 255

print("Nueva forma de X_train:", X_train.shape)
print("Nueva forma de X_test:", X_test.shape)

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

print("Ejemplo y_train[0]:", y_train[0])
print("Ejemplo one-hot:", y_train_cat[0])

## 3. Definir el modelo

Usaremos un modelo secuencial con:

- una capa de entrada de 784 variables,
- una capa oculta de 512 neuronas con activación ReLU,
- una capa de salida de 10 neuronas con activación softmax.

¿Por qué softmax?
Porque estamos resolviendo un problema de clasificación multiclase.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28 * 28,)),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax")
])

In [ ]:
model.summary()

### Cómo leer `model.summary()`

- **Output Shape**: tamaño de la salida de cada capa.
- **Param #**: número de parámetros entrenables.
- Más neuronas implica más parámetros, y por tanto mayor capacidad del modelo.

## 4. Compilar el modelo

Aquí definimos:

- el [optimizador](https://arxiv.org/pdf/1609.04747.pdf): cómo se actualizan los pesos,
- la función de pérdida: qué error se minimiza,
- las métricas: cómo reportaremos desempeño.

Usaremos:

- `adam` como optimizador,
- `categorical_crossentropy` como pérdida,
- `accuracy` como métrica.

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## 5. Entrenar el modelo

Usaremos `fit()` con:

- `epochs`: número de pasadas sobre los datos,
- `batch_size`: tamaño del mini-batch,
- `validation_split`: porcentaje del conjunto de entrenamiento reservado para validación.

La validación nos permite monitorear generalización durante el entrenamiento.

In [ ]:
history = model.fit(
    X_train,
    y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

## 6. Curvas de entrenamiento y validación

Ahora visualizaremos:

- la pérdida (`loss`),
- la exactitud (`accuracy`),

tanto en entrenamiento como en validación.

In [ ]:
history_dict = history.history

loss = history_dict["loss"]
val_loss = history_dict["val_loss"]
epochs = range(1, len(loss) + 1)

plt.figure(figsize=(6,4))
plt.plot(epochs, loss, label="Train loss")
plt.plot(epochs, val_loss, label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]

plt.figure(figsize=(6,4))
plt.plot(epochs, acc, label="Train accuracy")
plt.plot(epochs, val_acc, label="Val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Pregunta para discutir

- ¿Las curvas de entrenamiento y validación evolucionan de forma parecida?
- ¿Hay señales de sobreajuste?
- ¿Qué esperaría que ocurriera si entrenáramos muchas más épocas?

## 7. Evaluación final en test

Una vez entrenado el modelo, evaluamos su desempeño en datos no vistos.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

## 8. Predicción de ejemplos individuales

Veamos una predicción concreta del modelo.

In [ ]:
pred = model.predict(X_test[:1], verbose=0)

print("Probabilidades:", pred[0])
print("Clase predicha:", np.argmax(pred[0]))
print("Clase real:", y_test[0])

In [ ]:
plt.figure(figsize=(3,3))
plt.imshow(X_test[0].reshape(28,28), cmap="gray")
plt.title(f"Real: {y_test[0]} | Predicha: {np.argmax(pred[0])}")
plt.axis("off")
plt.show()

## 9. Resumen

En este notebook vimos el flujo básico de Keras:

1. cargar datos,
2. preprocesar,
3. definir el modelo,
4. compilar,
5. entrenar con `fit`,
6. evaluar en test,
7. predecir nuevos ejemplos.

Este es el esqueleto general que reutilizaremos en otros problemas de deep learning.

## Cambios que pueden probar

Prueba uno o más de los siguientes cambios:

- cambiar la cantidad de neuronas de la capa oculta,
- agregar una segunda capa oculta,
- cambiar el número de épocas,
- cambiar el batch size.

Luego compara el desempeño en validación y test.

## Cómo guardar y cargar un modelo

In [ ]:
model.save("mnist_mlp.keras")

loaded_model = keras.models.load_model("mnist_mlp.keras")

## Bonus opcional: el mismo ejemplo en PyTorch

A continuación se muestra una versión breve del mismo problema usando **PyTorch**.

El objetivo **no** es aprender PyTorch en detalle, sino notar que las ideas centrales son las mismas:

- definir una arquitectura,
- elegir una función de pérdida,
- elegir un optimizador,
- entrenar el modelo,
- evaluar su desempeño.

La principal diferencia es que en PyTorch el entrenamiento suele escribirse con un loop más explícito.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch:", torch.__version__)

### Preparar los datos

Usaremos el mismo preprocesamiento base del ejemplo en Keras:

- imágenes reescaladas a valores entre 0 y 1,
- cada imagen convertida a un vector de 784 entradas,
- etiquetas como enteros entre 0 y 9.

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape((60000, 28 * 28)).astype("float32") / 255
X_test = X_test.reshape((10000, 28 * 28)).astype("float32") / 255

# Convertir a tensores de PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(X_train_t.shape, y_train_t.shape)
print(X_test_t.shape, y_test_t.shape)

### Definir el modelo

La arquitectura será análoga a la usada en Keras:

- entrada de tamaño 784,
- una capa oculta de 512 neuronas con ReLU,
- una capa de salida de 10 neuronas.

In [ ]:
model_torch = nn.Sequential(
    nn.Linear(28 * 28, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
)

print(model_torch)

### Función de pérdida y optimizador

Usaremos:

- `CrossEntropyLoss` para clasificación multiclase,
- `Adam` como optimizador.

En PyTorch, `CrossEntropyLoss` ya incorpora internamente la operación necesaria sobre la salida, por lo que no agregamos `softmax` explícitamente en la última capa.

In [4]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_torch.parameters(), lr=0.001)

### Entrenamiento

Aquí aparece una diferencia importante con Keras:

- en Keras solemos usar `fit()`,
- en PyTorch escribimos el loop de entrenamiento de forma explícita.

In [ ]:
epochs = 3

for epoch in range(epochs):
    model_torch.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model_torch(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - Train loss: {epoch_loss:.4f}")

### Evaluación simple

In [ ]:
model_torch.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model_torch(xb)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

test_acc = correct / total
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
with torch.no_grad():
    sample_logits = model_torch(X_test_t[:1])
    sample_pred = torch.argmax(sample_logits, dim=1).item()

print("Clase predicha:", sample_pred)
print("Clase real:", y_test[0])

In [ ]:
plt.figure(figsize=(3,3))
plt.imshow(X_test[0].reshape(28,28), cmap="gray")
plt.title(f"Real: {y_test[0]} | Predicha: {sample_pred}")
plt.axis("off")
plt.show()

## Comparación conceptual: Keras vs PyTorch

### En Keras
- definimos el modelo,
- usamos `compile()`,
- usamos `fit()`.

### En PyTorch
- definimos el modelo,
- definimos la función de pérdida,
- definimos el optimizador,
- escribimos explícitamente el loop de entrenamiento.

## Idea importante

Aunque la sintaxis cambia, las ideas fundamentales son las mismas:

- arquitectura,
- pérdida,
- gradientes,
- actualización de parámetros,
- evaluación.

### Comentario final

Para este curso trabajaremos principalmente con **Keras**, porque permite concentrarnos mejor en los conceptos y en las aplicaciones con menos carga de programación.

PyTorch es otra herramienta muy importante y ampliamente usada, especialmente en investigación.